# MediSense-Cardio -- Photo-quality scores for the new 100 recordings (question C3)

**What this notebook does:** computes the **five photo-quality scores** (blur, skew, resolution, contrast, blockiness) for the
**1,300 pages of the new 100 recordings**: the clean page + 6 damage types x levels 3 and 5 = 13 versions per recording.
It is the same code, applied to the same pages, as the first-50 test, so the two sets of scores can be compared.

**Why:** question **C3** asks whether a detector trained on the first 50 recordings can warn about wrong heart rates on
recordings it has never seen. The analysis itself runs later on the laptop. This notebook only produces the scores.

**What it needs:** nothing new. The clean pages and `results_extension.csv` are already on Drive (study folder).
**No GPU and no digitiser.** About 15-25 minutes on Colab's CPU. **Resumable:** if it stops, press `Runtime -> Run all`; it continues
from the last saved record.

**Output:** `quality_signals_ext.csv` in the study folder (1,300 rows). The last cell prints `ALL SIGNALS COMPLETE`.
Download `quality_signals_ext.csv` and `quality_signals_v2.csv` from Drive when it finishes.


## Step 0 -- Mount Google Drive and check the study folder

**What this does:** connects to the study folder and checks that the files this notebook needs are there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
import numpy as np
import pandas as pd

STUDY_DIR = '/content/drive/MyDrive/MediSense-Cardio-Study'
CLEAN_DIR = f'{STUDY_DIR}/images_clean'
EXT_RESULTS = f'{STUDY_DIR}/results_extension.csv'
SIGNALS_EXT = f'{STUDY_DIR}/quality_signals_ext.csv'
SIGNALS_V2 = f'{STUDY_DIR}/quality_signals_v2.csv'

assert os.path.isdir(STUDY_DIR), 'Study folder not found: add the shortcut to My Drive first'
assert os.path.exists(EXT_RESULTS), 'results_extension.csv not found on Drive'
assert os.path.isdir(CLEAN_DIR), 'images_clean not found on Drive'
print('Study folder:', STUDY_DIR)
print('results_extension.csv found; clean pages on Drive:', len(glob.glob(f'{CLEAN_DIR}/*.png')))
print('quality_signals_v2.csv (first 50) found:', os.path.exists(SIGNALS_V2))


## Step 1 -- The 1,300 pages of the new 100 recordings

**What this does:** reads the list of versions from `results_extension.csv` (the file of the run that was digitised) and checks that every
recording has its clean page. Nothing is chosen here: the list is exactly the 1,300 images that were digitised.


In [ ]:
ext = pd.read_csv(EXT_RESULTS)
assert len(ext) == 1300 and ext['record'].nunique() == 100, 'unexpected results_extension.csv'
records = sorted(ext['record'].unique())
missing = [r for r in records if not os.path.exists(f'{CLEAN_DIR}/{r}-0.png')]
assert not missing, f'{len(missing)} clean pages missing on Drive: {missing[:5]}'
print(f'{len(records)} recordings, {len(ext)} versions to score; every clean page is on Drive.')
print(ext.groupby(['axis', 'level']).size().to_string())


## Step 2 -- Damage code and score code (copied word for word)

**What this does:** defines the six damage functions (copied from the Extension notebook, the code that made the pages the digitiser
saw) and the five-score function (copied from the earlier detector notebook that scored the first 50). Nothing is changed.


In [ ]:
import cv2
import numpy as np

import cv2

def degrade_blur(img, level):
    k = [3, 7, 11, 15, 21][level - 1]
    return cv2.GaussianBlur(img, (k, k), 0)

def degrade_rotate(img, level):
    angle = [2, 5, 8, 12, 16][level - 1]
    h, w = img.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))

def degrade_jpeg(img, level):
    quality = [50, 30, 15, 8, 3][level - 1]
    ok, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

def degrade_resolution(img, level):
    factor = [0.75, 0.5, 0.35, 0.25, 0.15][level - 1]
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def degrade_brightness(img, level):
    strength = [0.15, 0.3, 0.45, 0.6, 0.75][level - 1]
    h, w = img.shape[:2]
    gradient = np.tile(np.linspace(1.0, 1.0 - strength, w), (h, 1)).astype(np.float32)
    gradient = cv2.merge([gradient, gradient, gradient])
    return np.clip(img.astype(np.float32) * gradient, 0, 255).astype(np.uint8)

def degrade_grid_contrast(img, level):
    strength = [0.15, 0.3, 0.45, 0.6, 0.75][level - 1]
    mean_val = img.mean()
    return np.clip(img.astype(np.float32) * (1 - strength) + mean_val * strength, 0, 255).astype(np.uint8)

AXES = {
    'blur': degrade_blur,
    'rotate': degrade_rotate,
    'jpeg': degrade_jpeg,
    'resolution': degrade_resolution,
    'brightness': degrade_brightness,
    'gridcontrast': degrade_grid_contrast,
}
LEVELS = [3, 5]  # extension: middle and worst level only


def compute_signals(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    edges = cv2.Canny(gray, 50, 150)
    min_len = int(0.3 * gray.shape[1])
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=min_len, maxLineGap=10)
    if lines is None or len(lines) < 3:
        skew_deg = None
    else:
        arr = lines.reshape(-1, 4) if lines.ndim == 3 else lines
        devs = []
        for x1, y1, x2, y2 in arr:
            a = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
            d = a % 90
            devs.append(min(d, 90 - d))
        skew_deg = float(np.median(devs))
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
    mag = np.abs(f)
    h, w = mag.shape; cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2); r_max = min(cy, cx)
    resolution_score = float(mag[r > 0.5 * r_max].sum() / (mag.sum() + 1e-9))
    contrast_score = float(gray.std())
    col_grad = np.mean(np.abs(np.diff(gray.astype(np.float32), axis=1)), axis=0)
    col_grad = col_grad - col_grad.mean()
    spec = np.abs(np.fft.rfft(col_grad))
    if len(spec) < 4:
        blockiness_score = 0.0
    else:
        k0 = int(round(len(col_grad) / 8))
        band = spec[max(1, k0 - 1):k0 + 2].mean(); bg = np.median(spec[1:])
        blockiness_score = float(band / (bg + 1e-6))
    return dict(blur_score=blur_score, skew_deg=skew_deg, resolution_score=resolution_score,
                contrast_score=contrast_score, blockiness_score=blockiness_score)

print('damage functions:', list(AXES), '| levels used here:', LEVELS)


## Step 3 -- Compute the five scores for all 1,300 pages (resumable)

**What this does:** for each recording it loads the clean page, builds the 13 versions (clean + 6 damages x levels 3 and 5), computes the five
scores of each, and **saves to Drive after every 10 recordings**. If the session stops, run it again: finished recordings are skipped.


In [ ]:
import time

done = pd.read_csv(SIGNALS_EXT) if os.path.exists(SIGNALS_EXT) else pd.DataFrame()
done_records = set(done['record']) if len(done) else set()
print(f'{len(done_records)} recordings already scored on Drive; {len(records) - len(done_records)} to do.')

rows = done.to_dict('records') if len(done) else []
t0 = time.time(); n_new = 0
for i, rec in enumerate(records):
    if rec in done_records:
        continue
    img = cv2.imread(f'{CLEAN_DIR}/{rec}-0.png')
    assert img is not None, f'cannot read clean page of {rec}'
    rows.append(dict(variant_id=f'{rec}-0__clean0', record=rec, axis='clean', level=0, **compute_signals(img)))
    for axis, fn in AXES.items():
        for level in LEVELS:
            rows.append(dict(variant_id=f'{rec}-0__{axis}{level}', record=rec, axis=axis, level=level,
                             **compute_signals(fn(img, level))))
    n_new += 1
    if n_new % 10 == 0 or i == len(records) - 1:
        pd.DataFrame(rows).to_csv(SIGNALS_EXT, index=False)
        print(f'  {len(set(r["record"] for r in rows))}/{len(records)} recordings scored; saved; {time.time() - t0:.0f} s this session')
pd.DataFrame(rows).to_csv(SIGNALS_EXT, index=False)
print('Done with Step 3.')


## Step 4 -- Check: same code, same scores as the first 50

**What this does:** the first-50 scores are in `quality_signals_v2.csv` on Drive (from the earlier detector notebook). This cell re-computes a few of them
with the code above and checks that they agree, so we know the new and the old scores are comparable. If `quality_signals_v2.csv` is missing,
the next cell rebuilds it (about 20 minutes).


In [ ]:
if os.path.exists(SIGNALS_V2):
    old = pd.read_csv(SIGNALS_V2)
    print(f'quality_signals_v2.csv: {len(old)} rows, {old["record"].nunique()} recordings')
    FEATS = ['blur_score', 'skew_deg', 'resolution_score', 'contrast_score', 'blockiness_score']
    ALL_FUNCS = AXES
    checks = []
    for rec in sorted(old['record'].unique())[:3]:
        img = cv2.imread(f'{CLEAN_DIR}/{rec}-0.png')
        assert img is not None, f'first-50 clean page missing for {rec}'
        for axis, level in [('clean', 0), ('blur', 3), ('resolution', 5), ('rotate', 5)]:
            vid = f'{rec}-0__{axis}{level}'
            im = img if axis == 'clean' else ALL_FUNCS[axis](img, level)
            new = compute_signals(im)
            ref = old[old['variant_id'] == vid].iloc[0]
            for f in FEATS:
                a, b = new[f], ref[f]
                same = (a is None and pd.isna(b)) or (a is not None and not pd.isna(b) and abs(a - b) <= 1e-6 * max(1.0, abs(b)))
                checks.append(same)
                if not same:
                    print('MISMATCH', vid, f, a, b)
    print(f'{sum(checks)} of {len(checks)} re-computed scores equal the saved first-50 scores.')
    assert all(checks), 'new and old scores are NOT computed identically -- stop and tell Claude'
    SIGNALS_V2_READY = True
else:
    SIGNALS_V2_READY = False
    print('quality_signals_v2.csv not found: run the next cell to rebuild it.')


In [ ]:
# Only runs if quality_signals_v2.csv was missing. Rebuilds the first-50 scores (31 versions x 50 recordings), same code.
if not SIGNALS_V2_READY:
    v2 = pd.read_csv(f'{STUDY_DIR}/results_v2.csv')
    f50 = sorted(v2['record'].unique())
    assert len(f50) == 50
    ALL_LEVELS = [1, 2, 3, 4, 5]
    rows50 = []
    for i, rec in enumerate(f50):
        img = cv2.imread(f'{CLEAN_DIR}/{rec}-0.png')
        assert img is not None, f'cannot read clean page of {rec}'
        rows50.append(dict(variant_id=f'{rec}-0__clean0', record=rec, axis='clean', level=0, **compute_signals(img)))
        for axis, fn in AXES.items():
            for level in ALL_LEVELS:
                rows50.append(dict(variant_id=f'{rec}-0__{axis}{level}', record=rec, axis=axis, level=level, **compute_signals(fn(img, level))))
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/50 recordings done')
    pd.DataFrame(rows50).to_csv(SIGNALS_V2, index=False)
    print('Saved', SIGNALS_V2, len(rows50), 'rows')
else:
    print('Nothing to rebuild: quality_signals_v2.csv is already on Drive.')


## Step 5 -- Final check and what to download

**What this does:** joins the scores to `results_extension.csv` (every one of the 1,300 images must match), re-reads the saved file from Drive,
and prints `ALL SIGNALS COMPLETE`. Then download **`quality_signals_ext.csv`** and **`quality_signals_v2.csv`** from the study folder and
tell Claude.


In [ ]:
sig = pd.read_csv(SIGNALS_EXT)
m = ext.merge(sig, on='variant_id', how='inner', suffixes=('', '_sig'))
assert len(sig) == 1300 and sig['variant_id'].is_unique, f'expected 1,300 unique rows, got {len(sig)}'
assert len(m) == 1300, 'join with results_extension.csv failed'
print(f'Verified on Drive: {len(sig)} rows in quality_signals_ext.csv, all 1,300 match results_extension.csv.')
print('Missing skew values (too few lines found):', int(sig['skew_deg'].isna().sum()))
print(sig.groupby(['axis', 'level'])[['blur_score', 'resolution_score', 'contrast_score']].median().round(3).to_string())
print('\nALL SIGNALS COMPLETE -- download quality_signals_ext.csv and quality_signals_v2.csv from the study folder.')
